In [1]:
# Parameters
BATCH_MODE = True


# Générateur de Recettes PDF
## Introduction - receipe maker

**Navigation** : [Index](../README.md)

Ce notebook couvre les concepts et techniques principaux de receipe maker. Les objectifs pedagogiques incluent la comprehension des fondamentaux, la mise en pratique via des exercices, et analyse de resultats.

**Prerequis** : notions de base en Python et en algorithmique.



Ce notebook orchestre une collaboration multi-agents pour :
1. Collecter des contraintes utilisateur
2. Générer une recette personnalisée
3. Produire un PDF stylisé

**Agents** :  
- `InputCollector` → Collecte des préférences  
- `RecipeGenerator` → Crée la recette via LLM  
- `PDFGenerator` → Génère le document final

---

### Installation des bibliothèques

In [2]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')

# Cell 1 - Installation


## État Global & Configuration

In [3]:
class RecipeState:
    def __init__(self):
        self.diet: str = ""
        self.excluded_ingredients: list[str] = []
        self.guests: int = 4
        self.ingredients: list[str] = []
        self.steps: list[str] = []
        self.cooking_time: float = 0.0
        self.ready_to_generate: bool = False
        self.pdf_path: str = ""

print("Classe RecipeState définie")
print("Classe RecipeState definie")


Classe RecipeState définie
Classe RecipeState definie


### Exercice 1 : Enrichir l'etat partage avec des preferences culinaires

La classe `RecipeState` stocke le regime, les ingredients exclus et le nombre de convives. L'objectif est de l'enrichir avec de nouveaux champs : **niveau de difficulte** (debutant, intermediaire, expert), **temps de preparation maximum**, et **cuisine preferee** (francaise, italienne, asiatique, etc.).

**Objectif** : etendre `RecipeState` avec ces nouveaux attributs et creer les methodes de mise a jour correspondantes.

**Indices** :
- # Etape 1 : Ajouter les attributs `difficulty`, `max_prep_time` et `cuisine_type` dans `__init__`
- # Etape 2 : Creer les methodes `set_difficulty`, `set_max_prep_time` et `set_cuisine`
- # Indice : suivre le pattern existant des autres attributs (type hints + valeur par defaut)

In [1]:
class ExtendedRecipeState(RecipeState):
    # TODO etudiant : enrichir l'etat avec les nouvelles preferences
    
    def __init__(self):
        super().__init__()
        self.difficulty: str = ""         # Etape 1 : debutant / intermediaire / expert
        self.max_prep_time: float = 0.0   # Etape 1 : temps max en minutes
        self.cuisine_type: str = ""       # Etape 1 : type de cuisine preferee
    
    def set_difficulty(self, level: str) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_max_prep_time(self, minutes: float) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_cuisine(self, cuisine: str) -> str:
        result = None  # TODO etudiant : assigner
        return result  # TODO etudiant : retourner confirmation

print("Exercice a completer : ExtendedRecipeState")

Exercice a completer


### Plugins d'agents et fonctions de modification d'état

In [4]:
from semantic_kernel.functions import kernel_function
from reportlab.pdfgen import canvas

class InputCollectorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="set_diet", description="Définit le régime alimentaire")
    def set_diet(self, diet: str) -> str:
        self.state.diet = diet
        return f"Régime {diet} enregistré"

    @kernel_function(name="exclude_ingredient", description="Ajoute un ingrédient à exclure")
    def exclude_ingredient(self, ingredient: str) -> str:
        self.state.excluded_ingredients.append(ingredient)
        return f"Ingrédient {ingredient} exclu"

    @kernel_function(name="set_guests", description="Définit le nombre de convives")
    def set_guests(self, guests: int) -> str:
        self.state.guests = guests
        return f"{guests} convives prévus"

class RecipeGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state

    @kernel_function(name="submit_recipe", description="Valide la recette générée")
    def submit_recipe(self, ingredients: list[str], steps: list[str], cooking_time: float) -> str:
        self.state.ingredients = ingredients
        self.state.steps = steps
        self.state.cooking_time = cooking_time
        self.state.ready_to_generate = True
        return "Recette validée et prête pour la génération PDF"



class PDFGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="generate_pdf", description="Génère le PDF final")
    def generate_pdf(self, output_path: str) -> str:
        c = canvas.Canvas(output_path)
        c.drawString(100, 800, "Recette personnalisée")
        c.drawString(100, 780, f"Pour {self.state.guests} personnes")
        c.save()
        self.state.pdf_path = output_path
        return f"PDF généré : {output_path}"

print("Imports Semantic Kernel OK")


Imports Semantic Kernel OK


### Exercice 2 : Plugin d'estimation nutritionnelle

Les plugins actuels gerent la collecte d'informations et la generation de recette, mais aucune information nutritionnelle n'est fournie. L'objectif est d'ajouter un `NutritionPlugin` qui estime les calories et macronutriments d'une recette.

**Objectif** : creer une classe `NutritionPlugin` avec une methode `estimate_nutrition` annotee `@kernel_function` qui retourne une estimation des calories par portion.

**Indices** :
- # Etape 1 : Definir un dictionnaire approximatif de calories par ingredient de base
- # Etape 2 : Calculer la somme des calories et diviser par le nombre de convives
- # Indice : le dictionnaire peut contenir des valeurs moyennes (ex: poulet = 165 kcal/100g, riz = 130 kcal/100g)

In [1]:
class NutritionPlugin:
    """Plugin d'estimation nutritionnelle pour les recettes."""
    
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="estimate_nutrition", description="Estime les calories par portion")
    def estimate_nutrition(self, ingredients: str) -> str:
        # TODO etudiant : implementer l'estimation nutritionnelle
        # Etape 1 : dictionnaire de reference (ingredient -> kcal/100g)
        CALORIES_REF = {}  # TODO etudiant : completer le dictionnaire
        
        # Etape 2 : parser les ingredients et calculer les calories totales
        total_calories = 0  # TODO etudiant : calculer
        calories_par_portion = None  # TODO etudiant : diviser par state.guests
        
        result = None  # TODO etudiant : formater le resultat
        return result  # TODO etudiant : retourner la description nutritionnelle

print("Exercice a completer : NutritionPlugin")

Exercice a completer


### Création des agents

In [5]:
import os
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")


def build_kernel(plugin, plugin_name: str) -> Kernel:
    """Cree un kernel avec un service de chat OpenAI et le plugin fourni.

    Sans add_service, l'agent leve "No service found" des le premier appel LLM.
    """
    kernel = Kernel()
    kernel.add_service(
        OpenAIChatCompletion(
            service_id="openai",
            ai_model_id=CHAT_MODEL,
            api_key=OPENAI_API_KEY,
        )
    )
    kernel.add_plugin(plugin, plugin_name)
    return kernel


shared_state = RecipeState()

# InputCollector
input_kernel = build_kernel(InputCollectorPlugin(shared_state), "input_plugin")
input_agent = ChatCompletionAgent(
    kernel=input_kernel,
    name="InputCollector",
    instructions="""Collectez les informations utilisateur de manière structurée.
    - Demandez d'abord le régime alimentaire
    - Puis les ingrédients à exclure
    - Enfin le nombre de convives"""
)

# RecipeGenerator avec son plugin
recipe_kernel = build_kernel(RecipeGeneratorPlugin(shared_state), "recipe_plugin")
recipe_agent = ChatCompletionAgent(
    kernel=recipe_kernel,
    name="RecipeGenerator",
    instructions="""Générez des recettes en respectant les contraintes.
    - Convertir les ingrédients en liste Python
    - Structurer les étapes de préparation
    - Calculer le temps de cuisson"""
)

# PDFGenerator
pdf_kernel = build_kernel(PDFGeneratorPlugin(shared_state), "pdf_plugin")
pdf_agent = ChatCompletionAgent(
    kernel=pdf_kernel,
    name="PDFGenerator",
    instructions="""Générez un PDF professionnel:
    - Structurez en sections claires
    - Utilisez une mise en page aérée"""
)

print("Imports et configuration OK")


Imports et configuration OK


### création de la conversation avec critères de terminaison et/ou de sélection

In [6]:
from semantic_kernel.agents import AgentGroupChat
from pydantic import PrivateAttr

from semantic_kernel.agents.strategies import TerminationStrategy

class ReadyTerminationStrategy(TerminationStrategy):
    _state: RecipeState = PrivateAttr()

    def __init__(self, state: RecipeState, **kwargs):
        super().__init__(**kwargs)
        self._state = state

    async def should_agent_terminate(self, agent, history):
        return self._state.ready_to_generate


group_chat = AgentGroupChat(
    agents=[input_agent, recipe_agent, pdf_agent],
    termination_strategy=ReadyTerminationStrategy(shared_state),
)

print("Imports agents OK")


Imports agents OK


### Boucle principale

In [7]:
# Fallback: s'assurer que group_chat est défini
try:
    group_chat
except NameError:
    print("⚠️ group_chat non défini - vérifiez que la cellule précédente a été exécutée")
    group_chat = None

async def recipe_workflow():
    if group_chat is None:
        print("❌ Impossible d'exécuter: group_chat non initialisé")
        return
        
    recipe_query = (
        "Je souhaite une recette végétarienne pour 6 personnes, "
        "sans champignons ni produits laitiers."
    )
    
    group_chat.history.add_user_message(recipe_query)
    
    try:
        async for message in group_chat.invoke():
            print(f"[{message.name}] {message.content}")
            
            if shared_state.ready_to_generate:
                print("\\nTransition vers la génération PDF...")
                break
                
    except Exception as e:
        print(f"Erreur lors de la génération: {str(e)}")
    
    print("\\nProcessus terminé")

await recipe_workflow()


[InputCollector] Votre demande a bien été enregistrée. Voici un récapitulatif :

- **Régime alimentaire :** Végétarien
- **Ingrédients exclus :** Champignons, Produits laitiers
- **Nombre de convives :** 6

Je vais maintenant vous proposer une recette végétarienne adaptée à vos critères.


[RecipeGenerator] Voici une délicieuse recette végétarienne pour 6 personnes, sans champignons ni produits laitiers :

### Salade de Quinoa aux Légumes

#### Ingrédients
```python
ingredients = [
    "400g de quinoa",
    "1 courgette",
    "2 poivrons (rouge et jaune)",
    "1 boîte de pois chiches (400g)",
    "1 oignon",
    "2 gousses d'ail",
    "4 cuillères à soupe d'huile d'olive",
    "1 cuillère à café de cumin moulu",
    "1 cuillère à soupe de paprika",
    "Sel",
    "Poivre",
    "1 bouquet de coriandre fraîche"
]
```

#### Étapes de préparation
1. Rincer le quinoa sous l'eau froide, puis le cuire dans une casserole avec 800 ml d'eau bouillante pendant 15 minutes. Égoutter et laisser refroidir.
2. Dans une grande poêle, chauffer l'huile d'olive et ajouter l'oignon émincé et l'ail haché. Faire revenir jusqu'à ce qu'ils soient dorés.
3. Ajouter les poivrons coupés en dés et la courgette coupée en rondelles. Faire revenir pendant 5 à 7 minutes jusqu'à ce qu'ils soient tendres

### Exercice 3 : Validation des contraintes avant generation

Le workflow actuel lance la generation sans verifier que les contraintes sont coherentes. L'objectif est d'implementer une fonction `valider_contraintes` qui verifie la coherence des preferences utilisateur avant de les transmettre aux agents.

**Objectif** : ecrire une fonction qui detecte les incoherences (par exemple : regime vegane + demande de fromage, nombre de convives negatif, ingredients exclus qui font partie du regime).

**Indices** :
- # Etape 1 : Definir les regles de validation (liste de conditions a verifier)
- # Etape 2 : Retourner un tuple `(est_valide, liste_erreurs)` pour guider l'utilisateur
- # Indice : utiliser des conditions if/elif pour chaque regle et accumuler les messages d'erreur

In [1]:
def valider_contraintes(state: RecipeState) -> tuple[bool, list[str]]:
    # TODO etudiant : implementer la validation des contraintes
    erreurs = []
    
    # Etape 1 : verifier la coherence du regime et des ingredients exclus
    # Etape 2 : verifier le nombre de convives, la duree, etc.
    
    est_valide = len(erreurs) == 0  # TODO etudiant : ajouter les conditions
    return est_valide, erreurs

print("Exercice a completer : validation des contraintes")

Exercice a completer


### Génération de pdf

In [8]:
pass

print("Workflow prêt")
print("Workflow pret")


Workflow prêt
Workflow pret


In [9]:
if shared_state.pdf_path:
    print(f"PDF généré avec succès: {shared_state.pdf_path}")
    print("Contenu de la recette:")
    print(f"- Régime: {shared_state.diet}")
    print(f"- Ingrédients: {', '.join(shared_state.ingredients)}")
else:
    print("Échec de la génération:", 
          "Aucun PDF produit malgré les tentatives")


Échec de la génération: Aucun PDF produit malgré les tentatives


## Conclusion

Ce notebook a mis en oeuvre une collaboration multi-agents pour generer une recette personnalisee et produire un PDF.

**Points cles** :

- **Etat partage** : la classe `RecipeState` sert de memoire commune aux trois agents. Chaque plugin lit et ecrit dans cette instance unique, ce qui permet de transmettre les contraintes utilisateur jusqu'a la generation du PDF.
- **Plugins et `@kernel_function`** : le decorateur `@kernel_function` expose les methodes Python comme outils appelables par le LLM (`set_diet`, `submit_recipe`, `generate_pdf`).
- **Orchestration `AgentGroupChat`** : les agents `InputCollector`, `RecipeGenerator` et `PDFGenerator` dialoguent dans une conversation de groupe, chacun avec son propre kernel et son service de chat.
- **Strategie de terminaison** : `ReadyTerminationStrategy` interroge l'etat partage (`ready_to_generate`) pour arreter la boucle des que la recette est validee.
- **Configuration du service** : chaque kernel doit recevoir un service via `add_service` (ici `OpenAIChatCompletion`), faute de quoi l'invocation echoue avec `No service found`.

**Pour aller plus loin** :

- Enrichir le PDF avec une mise en page stylisee (sections, images, table des ingredients) via `reportlab.platypus`.
- Brancher une API nutritionnelle pour calculer les apports caloriques de la recette.
- Generer une illustration du plat avec un modele image et l'inserer dans le document.
